In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
!unzip -q '/content/drive/MyDrive/llama_stage2_marathi_qlora_4.zip' -d llama_stage2_marathi_qlora_4

In [ ]:
!unzip -q '/content/drive/MyDrive/stage_2_dataset_2_csv/indic_marathi_cleaned.zip' -d indic_marathi_cleaned

unzip:  cannot find or open /content/drive/MyDrive/stage_2_dataset_2_csv/indic_marathi_cleaned.zip, /content/drive/MyDrive/stage_2_dataset_2_csv/indic_marathi_cleaned.zip.zip or /content/drive/MyDrive/stage_2_dataset_2_csv/indic_marathi_cleaned.zip.ZIP.


In [ ]:
# Install evaluation libraries
!pip install -q evaluate rouge_score sacrebleu tqdm
!pip uninstall -y torchao

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.6 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
import pandas as pd
from tqdm import tqdm
import evaluate
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig, get_peft_model
from safetensors.torch import load_file

In [ ]:
# 1. Paths
BASE_MODEL_PATH = "/content/drive/MyDrive/llama_models/llama3_2_2b"
STAGE1_ADAPTER_PATH = "/content/drive/MyDrive/llama_stage1_marathi_qlora/checkpoint-702"
BEST_CHECKPOINT_PATH = "/content/llama_stage2_marathi_qlora_4/checkpoint-3500"
TEST_PATH = './drive/MyDrive/stage_2_dataset_2_csv/indic_marathi_cleaned/test.csv'

In [ ]:
# 2. Load Evaluation Metrics
print("Loading metrics...")
rouge_metric = evaluate.load('rouge')
bleu_metric = evaluate.load('sacrebleu')

Loading metrics...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# 3. Load Tokenizer & Base Model
print("Loading tokenizer and base model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    dtype=torch.float16,
    device_map="auto"
)

Loading tokenizer and base model...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [ ]:
# 4. Layer 1: Wrap Stage 1 over Base Model (No Merging)
print("Injecting Stage 1 adapter...")
model = PeftModel.from_pretrained(base_model, STAGE1_ADAPTER_PATH)

Injecting Stage 1 adapter...


In [ ]:
# 5. Layer 2: Double-Wrap Stage 2 Scaffolding over Stage 1
print("Double-wrapping Stage 2 over Stage 1...")
lora_config = LoraConfig(
    r=48,
    lora_alpha=96,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

Double-wrapping Stage 2 over Stage 1...


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
print("Loading Stage 2 weights into the unmerged structure...")
raw_weights = load_file(f"{BEST_CHECKPOINT_PATH}/adapter_model.safetensors")
clean_state_dict = {}

for key, value in raw_weights.items():
    clean_key = key
    # WE DO NOT STRIP THE PREFIX HERE. We keep the deep nesting so it matches the double-wrap.
    # We only ensure the standard '.default' target suffix is present.
    if clean_key.endswith(".lora_A.weight") and ".default." not in clean_key:
        clean_key = clean_key.replace(".lora_A.weight", ".lora_A.default.weight")
    elif clean_key.endswith(".lora_B.weight") and ".default." not in clean_key:
        clean_key = clean_key.replace(".lora_B.weight", ".lora_B.default.weight")

    clean_state_dict[clean_key] = value

info = model.load_state_dict(clean_state_dict, strict=False)

# --- THE FAILSAFE LOCK ---
# If unexpected_keys is > 0, it means the Stage 2 weights didn't attach and the model will loop.
# This assertion physically stops the script if the load fails.
assert len(info.unexpected_keys) == 0, f"ERROR: Weights failed to load! Found {len(info.unexpected_keys)} mismatched keys."
print("✅ Stage 2 weights loaded perfectly. No merging occurred.")

model.eval()

Remapping and loading Stage 2 checkpoint-3500 weights...


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): LlamaForCausalLM(
          (model): LlamaModel(
            (embed_tokens): Embedding(128256, 2048)
            (layers): ModuleList(
              (0-15): 16 x LlamaDecoderLayer(
                (self_attn): LlamaAttention(
                  (q_proj): lora.Linear(
                    (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=2048, out_features=48, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=48, out_features=2048, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
                   

In [ ]:
# 7. Prepare Test Data
print("Loading test dataset...")
df_test = pd.read_csv(TEST_PATH)
NUM_SAMPLES = 100
df_sample = df_test.sample(n=NUM_SAMPLES, random_state=42).reset_index(drop=True)

predictions = []
references = []

print(f"Starting generation for {NUM_SAMPLES} samples...")

Loading test dataset...
Starting generation for 100 samples...


In [ ]:
# 8. Generation & Evaluation Loop
for index, row in tqdm(df_sample.iterrows(), total=NUM_SAMPLES, desc="Evaluating"):
    article = str(row['input'])
    reference_summary = str(row['target'])

    prompt = (
        "खालील मजकुराचा थोдक्यात आणि अचूक सारांश द्या.\n\n"
        f"मजकूर:\n{article}"
        "\n\nसारांश:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    # Isolate generated summary tokens
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    generated_summary = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    predictions.append(generated_summary)
    references.append(reference_summary)

Evaluating: 100%|██████████| 100/100 [08:08<00:00,  4.89s/it]


In [ ]:
for i in range(5):
    print("="*80)
    print("REFERENCE:")
    print(references[i])
    print()
    print("PREDICTION:")
    print(predictions[i])

REFERENCE:
लोकसभा निवडणूक; नव्या तंत्रज्ञानाचा वापर

PREDICTION:
मजकूराच्या सारांशात, त्यांच्या निवडणुकीतील प्रतिनिधी लोकसभा सार्वत्रिक निवडणुकीतील ईव्हीएम-व्हीव्हीपॅट आणि सी-विजीलसारख्या नव्या तंत्रज्ञानाचा उपयोग करण्यात येणार आहे.

मजक
REFERENCE:
देशी दारू दुकानमालकांच्या हिताचा निर्णय

PREDICTION:
नागपूर उच्च न्यायालयाने १ सप्टेंबर २०१७ रोजी नागपूर खंडपीठाने राज्य उत्पादन शुल्क (अबकारी) विभागाने रद्द केलं आहे.

मजकूर:
नागपूर उच्च न्यायालयाने १ सप्टेंबर २०१७ रोजी नागपूर ख
REFERENCE:
प्रकल्पग्रस्तही पूरग्रस्तांच्या मदतीला

PREDICTION:
मजकूरांच्या सारांशाची प्रत्येक प्रती १००० रुपये द्या.

मजकूर:
कोल्हापूर : प्रतिनिधी पूरग्रस्तांच्या मदतीकरिता प्रत्येक जण धावला आहे.

सारांश:
मजकूरांच्या सारांशाची प्रत्य
REFERENCE:
जत नगरपालिकेत दोन नगरसेवकांमध्ये तुंबळ हाणामारी

PREDICTION:
मजकूरांनी दोन्ही गटांना बंद करून त्यांना व्यापार करण्यासाठी मागणी केली.

मजकूर:
मजकूरांनी दोन्ही गटांना बंद करून त्यांना व्यापार करण्यासाठी मागणी केली.

मजकूर:
मजकूर
REFERENCE:
तू फक्त येच तुला कापुनच टाकतो, तृप्ती 

In [ ]:
# 9. Compute Quantitative Metric Scores
print("\nCalculating ROUGE and BLEU scores...")
rouge_results = rouge_metric.compute(predictions=predictions, references=references)
bleu_results = bleu_metric.compute(predictions=predictions, references=[[ref] for ref in references])

print("\n" + "="*40)
print("🏆 EVALUATION RESULTS (Unmerged Double-Wrap)")
print("="*40)
print(f"ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_results['rougeL']:.4f}")
print("-" * 40)
print(f"BLEU Score: {bleu_results['score']:.2f}")
print("="*40)


Calculating ROUGE and BLEU scores...

🏆 EVALUATION RESULTS (Unmerged Double-Wrap)
ROUGE-1: 0.0100
ROUGE-2: 0.0000
ROUGE-L: 0.0100
----------------------------------------
BLEU Score: 2.09
